In [ ]:
import sys
import subprocess

def ensure_installed(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

ensure_installed(['transformers', 'datasets', 'torch', 'scikit-learn', 'pandas', 'numpy'])

In [ ]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, classification_report

if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    device_name = 'mps'
elif torch.cuda.is_available():
    device = torch.device('cuda')
    device_name = 'cuda'
else:
    device = torch.device('cpu')
    device_name = 'cpu'

print({'selected_device': device_name})

In [ ]:
dataset = load_dataset('dair-ai/emotion', split='test')
class_names = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

print({'split': 'test', 'num_rows': len(dataset)})
print(dataset[:3])

In [ ]:
model_name = 'nateraw/bert-base-uncased-emotion'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

id2label = {int(k): v.lower() for k, v in model.config.id2label.items()}
model_label_order = [id2label[i] for i in range(model.config.num_labels)]

print({
    'model_name': model_name,
    'num_labels': int(model.config.num_labels),
    'model_labels': model_label_order,
    'hidden_size': int(model.config.hidden_size)
})

In [ ]:
def predict_emotions(texts, batch_size=64, max_length=128):
    all_pred_ids = []
    all_confidences = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = model(**encoded)
            probs = torch.softmax(outputs.logits, dim=-1)
            pred_ids = torch.argmax(probs, dim=-1)
            confs = probs.gather(1, pred_ids.unsqueeze(1)).squeeze(1)

        all_pred_ids.append(pred_ids.cpu())
        all_confidences.append(confs.cpu())

    return torch.cat(all_pred_ids).numpy(), torch.cat(all_confidences).numpy()

In [ ]:
texts = dataset['text']
true_ids = np.array(dataset['label'])

pred_ids, pred_confidence = predict_emotions(texts, batch_size=64, max_length=128)
pred_labels = [class_names[i] for i in pred_ids]

results_df = pd.DataFrame({
    'text': texts,
    'true_label': [class_names[i] for i in true_ids],
    'predicted_label': pred_labels,
    'confidence': pred_confidence
})

print(results_df.head(10).to_dict(orient='records'))

In [ ]:
accuracy = accuracy_score(true_ids, pred_ids)
report = classification_report(true_ids, pred_ids, target_names=class_names, digits=4)

print({
    'model_name': model_name,
    'dataset': 'dair-ai/emotion',
    'split': 'test',
    'num_examples': len(dataset),
    'device': device_name,
    'accuracy': round(float(accuracy), 6)
})
print(report)

In [ ]:
sample_n = 8
print(results_df.head(sample_n).to_string(index=False))